# Activity 4 — PCA & the 2-D Map
**DSC 402/502 · Relationships.** Eight audio features per song is too many to see. PCA squeezes them into a 2-D scatter you *can* plot — but the new axes are **blends**, and what you *color* by may not explain where points land. Build the map, then interrogate it.

*Run cells top to bottom.*

In [ ]:
import pandas as pd, altair as alt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Top Spotify Songs 2023 (953 tracks). From Canvas, or loaded here from a mirror.
URL = 'https://raw.githubusercontent.com/zero-to-mastery/file-io/main/spotify-2023.csv'
try:
    df = pd.read_csv(URL, encoding='latin-1')      # latin-1 matters (non-UTF-8 track names)
except Exception:
    df = pd.read_csv('spotify-2023.csv', encoding='latin-1')  # fallback: upload the Canvas copy
print(len(df), 'songs')
df[['track_name','artist(s)_name','mode','bpm','energy_%']].head()

953 songs


,track_name,artist(s)_name,mode,bpm,energy_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",Major,125,83
1,LALA,Myke Towers,Major,92,74
2,vampire,Olivia Rodrigo,Major,138,53
3,Cruel Summer,Taylor Swift,Major,170,72
4,WHERE SHE GOES,Bad Bunny,Minor,144,80


### 1. Build the map
Reduce 8 features to 2 principal components (standardize first!), scatter, color by major/minor. Hovering shows the song — that’s the bonus, and it’s one line in Altair. **Look before theorizing:** do the colors form two clean regions, or mix all through?

In [ ]:
feats = ['bpm','danceability_%','valence_%','energy_%',
         'acousticness_%','instrumentalness_%','liveness_%','speechiness_%']
d = df.dropna(subset=feats+['mode']).copy()
for c in feats: d[c] = pd.to_numeric(d[c], errors='coerce')
d = d.dropna(subset=feats)

X = StandardScaler().fit_transform(d[feats])          # standardize
pca = PCA(n_components=2).fit(X)
pcs = pca.transform(X)
d['PC1'], d['PC2'] = pcs[:,0], pcs[:,1]

alt.Chart(d).mark_circle(size=60, opacity=0.6).encode(
    x='PC1:Q', y='PC2:Q', color='mode:N',
    tooltip=['track_name:N','artist(s)_name:N']).interactive()

alt.Chart(...)

### 2. Name the axes (the interpretability move)
**# YOUR TURN:** print the loadings and try to name PC1 and PC2 from their biggest contributors. One is easy to name, and one is hard. Which is which? and what does that reveals about how PCA helps or hurts?

In [ ]:
# YOUR TURN: loadings
loadings = pd.DataFrame(pca.components_.T, index=feats, columns=['PC1','PC2']).round(2)
loadings

,PC1,PC2
bpm,-0.02,-0.23
danceability_%,0.45,0.45
valence_%,0.46,0.24
energy_%,0.55,-0.37
acousticness_%,-0.48,0.38
instrumentalness_%,-0.16,-0.26
liveness_%,0.07,-0.38
speechiness_%,0.12,0.45


### 3. Does the color explain the picture?
You probably saw major/minor **doesn’t** cleanly separate. **# YOUR TURN:** recolor the *same* scatter by a continuous feature — `color='energy_%:Q'`. Does *that* line up with an axis? The layout is “about” whatever drives the **position**, which may be unrelated to **color**.

In [ ]:
# YOUR TURN: recolor by a continuous feature, try different size, opacity, etc.
alt.Chart(d).mark_circle(size=60, opacity=0.6).encode(
     x='PC1:Q', y='PC2:Q', color='energy_%:Q',
     tooltip=['track_name:N','artist(s)_name:N']).interactive()


alt.Chart(...)

### 4. Scaled vs. unscaled (compare)
**# YOUR TURN:** re-run PCA **without** `StandardScaler` and plot again. The axes and spread change — whichever features have the biggest raw numbers start to dominate. Which version would you publish, and why?

In [ ]:
# YOUR TURN: PCA on d[feats] directly (no StandardScaler) and re-plot
pca = PCA(n_components=2).fit(d[feats])
pcs = pca.transform(d[feats])
d['PC1'], d['PC2'] = pcs[:,0], pcs[:,1]

alt.Chart(d).mark_circle(size=60, opacity=0.6).encode(
    x='PC1:Q', y='PC2:Q', color='mode:N',
    tooltip=['track_name:N','artist(s)_name:N']).interactive()

alt.Chart(...)